Cell 1：clone 官方 MISO

In [2]:
# ============================================================
# Cell 1
# Clone official MISO repository
# ============================================================

from pathlib import Path
import shutil
import subprocess
import sys


WORK_ROOT = Path("/kaggle/working")

MISO_ROOT = (
    WORK_ROOT
    / "miso"
)

MISO_OUTPUT_ROOT = (
    WORK_ROOT
    / "MISO_baseline"
)

DATA_ROOT = Path(
    "/kaggle/input/datasets/wuvdji/smgc-data"
)


print("=" * 100)
print("RESET MISO WORKSPACE")
print("=" * 100)


if MISO_ROOT.exists():

    shutil.rmtree(
        MISO_ROOT
    )


if MISO_OUTPUT_ROOT.exists():

    shutil.rmtree(
        MISO_OUTPUT_ROOT
    )


for name in list(sys.modules):

    if (
        name == "miso"
        or name.startswith("miso.")
    ):

        del sys.modules[name]


subprocess.run(
    [
        "git",
        "clone",
        "https://github.com/kpcoleman/miso.git",
        str(MISO_ROOT),
    ],
    check=True,
)


assert MISO_ROOT.exists()
assert DATA_ROOT.exists()


MISO_COMMIT = (
    subprocess.check_output(
        [
            "git",
            "-C",
            str(MISO_ROOT),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    )
    .strip()
)


print(
    "MISO root :",
    MISO_ROOT
)

print(
    "Data root :",
    DATA_ROOT
)

print(
    "Git commit:",
    MISO_COMMIT
)

print(
    "\nPASS: official MISO source cloned."
)

RESET MISO WORKSPACE


Cloning into '/kaggle/working/miso'...


MISO root : /kaggle/working/miso
Data root : /kaggle/input/datasets/wuvdji/smgc-data
Git commit: 1252c6f4b280a0fd303c2a3e131e68b379acb3a6

PASS: official MISO source cloned.


Filtering content: 100% (2/2), 1.02 GiB | 24.66 MiB/s, done.


Cell 2：安装 Python 3.12 兼容依赖

In [3]:
# ============================================================
# Cell 2
# Install Python-3.12-compatible core dependencies
#
# Do NOT pip install official MISO package:
# setup.py requires Python == 3.7.*
# ============================================================

import sys
import subprocess


print("=" * 100)
print("INSTALL MISO COMPATIBLE DEPENDENCIES")
print("=" * 100)


subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--prefer-binary",

        "anndata==0.11.4",
        "scanpy==1.11.4",
        "scikit-learn==1.6.1",

        "tqdm",
        "Pillow",
    ],
    check=True,
)


print(
    "PASS: compatible core dependencies installed."
)

INSTALL MISO COMPATIBLE DEPENDENCIES
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.5 MB/s eta 0:00:00
PASS: compatible core dependencies installed.


Cell 3：环境兼容 + 官方源码审计

In [4]:
# ============================================================
# Cell 3
# MISO environment + source audit
# ============================================================

import sys
import importlib
import inspect

import numpy as np
import scipy
import scipy.sparse as sp
import scanpy as sc
import anndata
import sklearn
import torch


# ------------------------------------------------------------
# SciPy sparse .A compatibility
# Official MISO preprocess uses adata.X.A
# ------------------------------------------------------------

if not hasattr(
    sp.spmatrix,
    "A",
):

    sp.spmatrix.A = property(
        lambda self:
            self.toarray()
    )


for cls in [
    sp.csr_matrix,
    sp.csc_matrix,
    sp.coo_matrix,
]:

    if not hasattr(
        cls,
        "A",
    ):

        cls.A = property(
            lambda self:
                self.toarray()
        )


# ------------------------------------------------------------
# Import official source directly
# ------------------------------------------------------------

repo_path = str(
    MISO_ROOT
)

sys.path = [
    p
    for p in sys.path
    if p != repo_path
]

sys.path.insert(
    0,
    repo_path,
)

importlib.invalidate_caches()


from miso import Miso
from miso.utils import preprocess


print("=" * 100)
print("MISO ENVIRONMENT AUDIT")
print("=" * 100)

print(
    "Python        :",
    sys.version.split()[0]
)

print(
    "PyTorch       :",
    torch.__version__
)

print(
    "CUDA runtime  :",
    torch.version.cuda
)

print(
    "CUDA available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU           :",
        torch.cuda.get_device_name(0)
    )


print(
    "NumPy         :",
    np.__version__
)

print(
    "SciPy         :",
    scipy.__version__
)

print(
    "Scanpy        :",
    sc.__version__
)

print(
    "AnnData       :",
    anndata.__version__
)

print(
    "sklearn       :",
    sklearn.__version__
)


print(
    "\nMiso.__init__:"
)

print(
    inspect.signature(
        Miso.__init__
    )
)


print(
    "\nOfficial preprocess source:"
)

print(
    inspect.getsource(
        preprocess
    )
)


print(
    "\nOfficial cluster source:"
)

print(
    inspect.getsource(
        Miso.cluster
    )
)


assert torch.cuda.is_available()


print(
    "\nPASS: MISO official source imported."
)

print(
    "PASS: Python 3.12 compatibility layer active."
)

MISO ENVIRONMENT AUDIT
Python        : 3.12.13
PyTorch       : 2.10.0+cu128
CUDA runtime  : 12.8
CUDA available: True
GPU           : Tesla T4
NumPy         : 2.0.2
SciPy         : 1.16.3
Scanpy        : 1.11.4
AnnData       : 0.11.4
sklearn       : 1.6.1

Miso.__init__:
(self, features, ind_views='all', combs='all', sparse=False, neighbors=None, device='cpu')

Official preprocess source:
def preprocess(adata,modality):
  adata.var_names_make_unique()
  if modality in ['rna','atac']:
    sc.pp.filter_genes(adata,min_cells=10)
    sc.pp.log1p(adata)

    if scipy.sparse.issparse(adata.X):
      return adata.X.A
    else:
      return adata.X

  elif modality=='protein':
    adata.X = np.apply_along_axis(protein_norm, 1, (adata.X.A if scipy.sparse.issparse(adata.X) else np.array(adata.X)))
    return adata.X     

  elif modality=='metabolite':
    sc.pp.log1p(adata)
    if scipy.sparse.issparse(adata.X):
      return adata.X.A
    else:
      return adata.X


Official cluster source:
  

Cell 4：5 个数据集 loader + 官方 MISO preprocessing

In [5]:
# ============================================================
# Cell 4
# Dataset loader + official MISO preprocessing
# ============================================================

import numpy as np
import scanpy as sc


DATASET_SPECS = {

    "HLN-A1": {

        "rna":
            DATA_ROOT
            / "Human_Lymph_Nodes/A1/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "Human_Lymph_Nodes/A1/adata_ADT.h5ad",

        "second_type":
            "protein",

        "second_name":
            "ADT",

        "K":
            10,

        "n_spots":
            3484,
    },


    "HLN-D1": {

        "rna":
            DATA_ROOT
            / "Human_Lymph_Nodes/D1/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "Human_Lymph_Nodes/D1/adata_ADT.h5ad",

        "second_type":
            "protein",

        "second_name":
            "ADT",

        "K":
            11,

        "n_spots":
            3359,
    },


    "E18.5": {

        "rna":
            DATA_ROOT
            / "E18.5_mouse_brain/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "E18.5_mouse_brain/adata_ATAC.h5ad",

        "second_type":
            "atac",

        "second_name":
            "ATAC",

        "K":
            14,

        "n_spots":
            2129,
    },


    "S2-E15": {

        "rna":
            DATA_ROOT
            / "Mouse_Embryos_S2/E15/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "Mouse_Embryos_S2/E15/adata_ATAC.h5ad",

        "second_type":
            "atac",

        "second_name":
            "ATAC",

        "K":
            15,

        "n_spots":
            1939,
    },


    "S2-E18": {

        "rna":
            DATA_ROOT
            / "Mouse_Embryos_S2/E18/adata_RNA.h5ad",

        "second":
            DATA_ROOT
            / "Mouse_Embryos_S2/E18/adata_ATAC.h5ad",

        "second_type":
            "atac",

        "second_name":
            "ATAC",

        "K":
            16,

        "n_spots":
            2248,
    },
}


def load_miso_dataset(
    dataset_name,
):

    spec = DATASET_SPECS[
        dataset_name
    ]


    assert spec["rna"].exists()
    assert spec["second"].exists()


    rna = sc.read_h5ad(
        spec["rna"]
    )

    second = sc.read_h5ad(
        spec["second"]
    )


    assert (
        rna.n_obs
        == spec["n_spots"]
    )

    assert (
        second.n_obs
        == spec["n_spots"]
    )


    assert np.array_equal(
        np.asarray(
            rna.obs_names
        ),
        np.asarray(
            second.obs_names
        ),
    )


    # ========================================================
    # E18.5 metadata adapter
    # ========================================================

    if dataset_name == "E18.5":

        coords = np.column_stack(
            [
                np.asarray(
                    rna.obs[
                        "array_col"
                    ],
                    dtype=np.float64,
                ),

                np.asarray(
                    rna.obs[
                        "array_row"
                    ],
                    dtype=np.float64,
                ),
            ]
        )


        labels = (
            rna.obs[
                "Combined_Clusters_annotation"
            ]
            .astype(str)
            .to_numpy()
        )


        rna.obsm[
            "spatial"
        ] = coords.copy()


        rna.obs[
            "Spatial_Label"
        ] = labels.copy()


    # ========================================================
    # Benchmark metadata audit
    # ========================================================

    assert (
        "Spatial_Label"
        in rna.obs.columns
    )

    assert (
        "spatial"
        in rna.obsm
    )


    gt = (
        rna.obs[
            "Spatial_Label"
        ]
        .astype(str)
        .to_numpy()
    )


    coords = np.asarray(
        rna.obsm[
            "spatial"
        ],
        dtype=np.float64,
    )


    spot_ids = np.asarray(
        rna.obs_names.astype(str)
    )


    assert (
        len(
            np.unique(gt)
        )
        == spec["K"]
    )

    assert (
        coords.shape
        == (
            spec["n_spots"],
            2,
        )
    )


    # ========================================================
    # Official MISO preprocessing
    #
    # RNA / ATAC:
    #   filter_genes(min_cells=10)
    #   log1p
    #
    # protein:
    #   official protein_norm
    # ========================================================

    rna_pp = rna.copy()

    second_pp = second.copy()


    X_rna = preprocess(
        rna_pp,
        modality="rna",
    )


    X_second = preprocess(
        second_pp,
        modality=spec[
            "second_type"
        ],
    )


    X_rna = np.asarray(
        X_rna,
        dtype=np.float32,
    )


    X_second = np.asarray(
        X_second,
        dtype=np.float32,
    )


    # ========================================================
    # Population must remain frozen
    # ========================================================

    assert (
        X_rna.shape[0]
        == spec["n_spots"]
    )

    assert (
        X_second.shape[0]
        == spec["n_spots"]
    )


    assert np.isfinite(
        X_rna
    ).all()


    assert np.isfinite(
        X_second
    ).all()


    print(
        f"{dataset_name}: "
        f"{spec['n_spots']} spots | "
        f"K={spec['K']} | "
        f"RNA + {spec['second_name']}"
    )


    print(
        "RNA features after MISO preprocess:",
        X_rna.shape[1]
    )

    print(
        f"{spec['second_name']} features "
        f"after MISO preprocess:",
        X_second.shape[1]
    )


    return {
        "X_rna":
            X_rna,

        "X_second":
            X_second,

        "gt":
            gt,

        "coords":
            coords,

        "spot_ids":
            spot_ids,

        "spec":
            spec,
    }


print(
    "PASS: MISO dataset loader defined."
)

PASS: MISO dataset loader defined.


Cell 5：随机种子 + 官方 KMeans 兼容

In [6]:
# ============================================================
# Cell 5
# MISO reproducibility + official KMeans compatibility
# ============================================================

import random

import numpy as np
import torch

from sklearn.cluster import KMeans


DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


def set_miso_seed(
    seed,
):

    seed = int(seed)

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed(
            seed
        )

        torch.cuda.manual_seed_all(
            seed
        )


    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


def miso_official_cluster_compat(
    embedding,
    n_clusters,
):

    # Official MISO source:
    # KMeans(n_clusters, random_state=100)
    #
    # MISO pins sklearn==1.0.2, whose historical
    # default n_init was 10.
    #
    # Explicit n_init=10 preserves that behavior
    # under sklearn 1.6.x.

    km = KMeans(
        n_clusters=int(
            n_clusters
        ),
        random_state=100,
        n_init=10,
    )


    return km.fit_predict(
        embedding
    )


print(
    "Device:",
    DEVICE
)

print(
    "PASS: MISO seed helper defined."
)

print(
    "PASS: KMeans compatibility = "
    "random_state=100, n_init=10."
)

Device: cuda
PASS: MISO seed helper defined.
PASS: KMeans compatibility = random_state=100, n_init=10.


Cell 6：MISO 单次 runner

In [7]:
# ============================================================
# Cell 6
# MISO single-run benchmark runner
# ============================================================

from pathlib import Path
import gc
import json
import time

import numpy as np
import torch

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


MISO_OUTPUT_ROOT = Path(
    "/kaggle/working/MISO_baseline"
)


def dataset_folder_name(
    dataset_name,
):

    return {

        "HLN-A1": "HLNA1",
        "HLN-D1": "HLND1",
        "E18.5": "E185",
        "S2-E15": "S2E15",
        "S2-E18": "S2E18",

    }[
        dataset_name
    ]


def run_miso_once(
    dataset_name,
    seed,
    stage="smoke",
):

    print(
        "\n" + "=" * 110
    )

    print(
        f"MISO | "
        f"{dataset_name} | "
        f"seed={seed}"
    )

    print(
        "=" * 110
    )


    total_start = time.time()


    # ========================================================
    # Load + official preprocessing
    # ========================================================

    prep_start = time.time()


    data = load_miso_dataset(
        dataset_name
    )


    preprocessing_seconds = (
        time.time()
        - prep_start
    )


    spec = data[
        "spec"
    ]


    print(
        "\nInput spots :",
        spec["n_spots"]
    )

    print(
        "Target K    :",
        spec["K"]
    )

    print(
        "Modalities  :",
        f"RNA + {spec['second_name']}"
    )

    print(
        "MISO sparse : False"
    )

    print(
        "MISO views  : all"
    )

    print(
        "MISO combs  : all"
    )


    # ========================================================
    # Training seed
    # ========================================================

    set_miso_seed(
        seed
    )


    # ========================================================
    # Official MISO model
    # ========================================================

    train_start = time.time()


    model = Miso(

        [
            data[
                "X_rna"
            ],

            data[
                "X_second"
            ],
        ],

        ind_views="all",

        combs="all",

        sparse=False,

        device=DEVICE,
    )


    model.train()


    training_seconds = (
        time.time()
        - train_start
    )


    embedding = np.asarray(
        model.emb,
        dtype=np.float32,
    )


    # ========================================================
    # Structural audit
    # ========================================================

    assert (
        embedding.shape[0]
        == spec[
            "n_spots"
        ]
    )


    # For exactly 2 modalities:
    # 32 + 32 + 32 interaction = 96
    assert (
        embedding.shape[1]
        == 96
    ), embedding.shape


    assert np.isfinite(
        embedding
    ).all()


    # ========================================================
    # Official MISO clustering semantics
    # ========================================================

    pred = (
        miso_official_cluster_compat(

            embedding,

            spec["K"],
        )
    )


    assert (
        len(pred)
        == spec[
            "n_spots"
        ]
    )


    assert (
        len(
            np.unique(
                pred
            )
        )
        == spec["K"]
    )


    # ========================================================
    # Unified benchmark metrics
    # ========================================================

    ari = adjusted_rand_score(
        data["gt"],
        pred,
    )


    nmi = (
        normalized_mutual_info_score(
            data["gt"],
            pred,
            average_method="max",
        )
    )


    total_seconds = (
        time.time()
        - total_start
    )


    # ========================================================
    # Save
    # ========================================================

    out_dir = (
        MISO_OUTPUT_ROOT
        / stage
        / (
            dataset_folder_name(
                dataset_name
            )
            + f"_seed{seed}"
        )
    )


    out_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    np.save(
        out_dir
        / "embedding.npy",
        embedding,
    )


    np.save(
        out_dir
        / "pred_labels.npy",
        pred,
    )


    np.save(
        out_dir
        / "gt_labels.npy",
        data["gt"],
    )


    np.save(
        out_dir
        / "coords.npy",
        data["coords"],
    )


    np.save(
        out_dir
        / "spot_ids.npy",
        data["spot_ids"],
    )


    metrics = {

        "method":
            "MISO",

        "dataset":
            dataset_name,

        "training_seed":
            int(seed),

        "n_spots":
            int(
                spec[
                    "n_spots"
                ]
            ),

        "target_K":
            int(
                spec["K"]
            ),

        "predicted_K":
            int(
                len(
                    np.unique(
                        pred
                    )
                )
            ),

        "modalities":
            (
                "RNA+"
                + spec[
                    "second_name"
                ]
            ),

        "RNA_features_after_preprocess":
            int(
                data[
                    "X_rna"
                ].shape[1]
            ),

        "second_features_after_preprocess":
            int(
                data[
                    "X_second"
                ].shape[1]
            ),

        "preprocess_RNA":
            (
                "filter_genes(min_cells=10)"
                " + log1p"
            ),

        "preprocess_second":
            (
                "protein_norm"
                if spec[
                    "second_type"
                ]
                == "protein"
                else
                "filter_genes(min_cells=10)"
                " + log1p"
            ),

        "ind_views":
            "all",

        "combs":
            "all",

        "sparse":
            False,

        "epochs_per_modality":
            1000,

        "learning_rate":
            0.001,

        "embedding_dim":
            int(
                embedding.shape[1]
            ),

        "cluster_method":
            "KMeans",

        "cluster_random_state":
            100,

        "cluster_n_init":
            10,

        "ARI":
            float(
                ari
            ),

        "NMI":
            float(
                nmi
            ),

        "NMI_average_method":
            "max",

        "preprocessing_seconds":
            float(
                preprocessing_seconds
            ),

        "training_seconds":
            float(
                training_seconds
            ),

        "total_seconds":
            float(
                total_seconds
            ),

        "MISO_git_commit":
            MISO_COMMIT,
    }


    with (
        out_dir
        / "metrics.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            metrics,
            f,
            indent=2,
        )


    print(
        "\n" + "=" * 110
    )

    print(
        f"{dataset_name} MISO RESULT"
    )

    print(
        "=" * 110
    )


    print(
        "Embedding shape:",
        embedding.shape
    )

    print(
        "Predicted K    :",
        len(
            np.unique(
                pred
            )
        )
    )

    print(
        f"ARI = {ari:.12f}"
    )

    print(
        f"NMI = {nmi:.12f}"
    )

    print(
        f"Preprocessing = "
        f"{preprocessing_seconds:.2f}s"
    )

    print(
        f"Training      = "
        f"{training_seconds:.2f}s"
    )

    print(
        f"Total runtime = "
        f"{total_seconds:.2f}s"
    )

    print(
        "\nSaved:",
        out_dir
    )


    print(
        "\nPASS: MISO run completed."
    )

    print(
        f"PASS: "
        f"{spec['n_spots']}/"
        f"{spec['n_spots']} "
        f"spots evaluated."
    )


    # ========================================================
    # Cleanup
    # ========================================================

    del model
    del data

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return metrics

Cell 7：HLN-A1 seed0 smoke

In [8]:
# ============================================================
# Cell 7
# MISO HLN-A1 seed0 smoke
# ============================================================

HLNA1_SMOKE = run_miso_once(
    dataset_name="HLN-A1",
    seed=0,
    stage="smoke",
)


print(
    "\n" + "=" * 100
)

print(
    "HLN-A1 MISO SMOKE COMPLETE"
)

print(
    "=" * 100
)

print(
    "ARI:",
    HLNA1_SMOKE[
        "ARI"
    ]
)

print(
    "NMI:",
    HLNA1_SMOKE[
        "NMI"
    ]
)


MISO | HLN-A1 | seed=0


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


HLN-A1: 3484 spots | K=10 | RNA + ADT
RNA features after MISO preprocess: 17954
ADT features after MISO preprocess: 31

Input spots : 3484
Target K    : 10
Modalities  : RNA + ADT
MISO sparse : False
MISO views  : all
MISO combs  : all


Training network for modality 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Training network for modality 2:   0%|          | 0/1000 [00:00<?, ?it/s]


HLN-A1 MISO RESULT
Embedding shape: (3484, 96)
Predicted K    : 10
ARI = 0.270224646477
NMI = 0.340933005246
Preprocessing = 2.39s
Training      = 42.10s
Total runtime = 44.81s

Saved: /kaggle/working/MISO_baseline/smoke/HLNA1_seed0

PASS: MISO run completed.
PASS: 3484/3484 spots evaluated.

HLN-A1 MISO SMOKE COMPLETE
ARI: 0.2702246464772572
NMI: 0.3409330052457166


Cell 8：E18.5 seed0 smoke

In [9]:
# ============================================================
# Cell 8
# MISO E18.5 seed0 smoke
# ============================================================

E185_SMOKE = run_miso_once(
    dataset_name="E18.5",
    seed=0,
    stage="smoke",
)


print(
    "\n" + "=" * 100
)

print(
    "E18.5 MISO SMOKE COMPLETE"
)

print(
    "=" * 100
)

print(
    "ARI:",
    E185_SMOKE[
        "ARI"
    ]
)

print(
    "NMI:",
    E185_SMOKE[
        "NMI"
    ]
)


MISO | E18.5 | seed=0
E18.5: 2129 spots | K=14 | RNA + ATAC
RNA features after MISO preprocess: 20755
ATAC features after MISO preprocess: 158701

Input spots : 2129
Target K    : 14
Modalities  : RNA + ATAC
MISO sparse : False
MISO views  : all
MISO combs  : all


Training network for modality 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Training network for modality 2:   0%|          | 0/1000 [00:00<?, ?it/s]


E18.5 MISO RESULT
Embedding shape: (2129, 96)
Predicted K    : 14
ARI = 0.165081512285
NMI = 0.262146583336
Preprocessing = 13.16s
Training      = 64.52s
Total runtime = 77.91s

Saved: /kaggle/working/MISO_baseline/smoke/E185_seed0

PASS: MISO run completed.
PASS: 2129/2129 spots evaluated.

E18.5 MISO SMOKE COMPLETE
ARI: 0.16508151228477141
NMI: 0.26214658333591284


Cell 9：MISO FORMAL 5×10

In [10]:
# ============================================================
# Cell 9
# MISO FORMAL
# 5 datasets × seeds 0-9 = 50 runs
#
# Resume supported:
# valid completed runs are automatically skipped
#
# Smoke runs are NOT reused.
# ============================================================

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


FORMAL_ROOT = (
    MISO_OUTPUT_ROOT
    / "formal_10seeds"
)

FORMAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


DATASET_ORDER = [
    "HLN-A1",
    "HLN-D1",
    "E18.5",
    "S2-E15",
    "S2-E18",
]


EXPECTED = {

    "HLN-A1": {
        "n_spots": 3484,
        "K": 10,
    },

    "HLN-D1": {
        "n_spots": 3359,
        "K": 11,
    },

    "E18.5": {
        "n_spots": 2129,
        "K": 14,
    },

    "S2-E15": {
        "n_spots": 1939,
        "K": 15,
    },

    "S2-E18": {
        "n_spots": 2248,
        "K": 16,
    },
}


# ============================================================
# Frozen protocol
# ============================================================

PROTOCOL = {

    "method":
        "MISO",

    "datasets":
        DATASET_ORDER,

    "training_seeds":
        list(range(10)),

    "n_runs_per_dataset":
        10,

    "total_runs":
        50,

    "views":
        "all",

    "combinations":
        "all",

    "sparse":
        False,

    "epochs_per_modality":
        1000,

    "learning_rate":
        0.001,

    "latent_dim_per_view":
        32,

    "two_modality_embedding_dim":
        96,

    "RNA_preprocessing":
        "filter_genes(min_cells=10) + log1p",

    "ATAC_preprocessing":
        "filter_genes(min_cells=10) + log1p",

    "protein_preprocessing":
        "official MISO protein_norm",

    "clustering":
        "KMeans",

    "KMeans_random_state":
        100,

    "KMeans_n_init":
        10,

    "KMeans_note":
        (
            "Official MISO source uses "
            "KMeans(n_clusters, random_state=100). "
            "Official environment pins sklearn==1.0.2, "
            "whose default n_init was 10. "
            "n_init=10 is explicit here for compatibility."
        ),

    "ARI":
        "sklearn.metrics.adjusted_rand_score",

    "NMI":
        (
            "sklearn.metrics."
            "normalized_mutual_info_score"
            "(average_method='max')"
        ),

    "std":
        "population std, ddof=0",

    "benchmark_population":
        "all benchmark spots retained",

    "MISO_git_commit":
        MISO_COMMIT,
}


with (
    FORMAL_ROOT
    / "protocol.json"
).open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        PROTOCOL,
        f,
        indent=2,
    )


# ============================================================
# Existing-run validator
# ============================================================

def formal_run_dir(
    dataset,
    seed,
):

    return (
        FORMAL_ROOT
        / (
            dataset_folder_name(
                dataset
            )
            + f"_seed{seed}"
        )
    )


def valid_existing_run(
    dataset,
    seed,
):

    run_dir = formal_run_dir(
        dataset,
        seed,
    )


    required = {

        "metrics":
            run_dir
            / "metrics.json",

        "embedding":
            run_dir
            / "embedding.npy",

        "pred":
            run_dir
            / "pred_labels.npy",

        "gt":
            run_dir
            / "gt_labels.npy",

        "coords":
            run_dir
            / "coords.npy",

        "spot_ids":
            run_dir
            / "spot_ids.npy",
    }


    if not all(
        p.exists()
        for p in required.values()
    ):

        return False


    try:

        with required[
            "metrics"
        ].open(
            "r",
            encoding="utf-8",
        ) as f:

            m = json.load(f)


        exp = EXPECTED[
            dataset
        ]


        # ----------------------------------------------------
        # Metadata
        # ----------------------------------------------------

        if (
            m.get("method")
            != "MISO"
        ):

            return False


        if (
            m.get("dataset")
            != dataset
        ):

            return False


        if (
            int(
                m.get(
                    "training_seed",
                    -1,
                )
            )
            != seed
        ):

            return False


        if (
            int(
                m.get(
                    "n_spots",
                    -1,
                )
            )
            != exp[
                "n_spots"
            ]
        ):

            return False


        if (
            int(
                m.get(
                    "target_K",
                    -1,
                )
            )
            != exp["K"]
        ):

            return False


        if (
            int(
                m.get(
                    "predicted_K",
                    -1,
                )
            )
            != exp["K"]
        ):

            return False


        if (
            int(
                m.get(
                    "embedding_dim",
                    -1,
                )
            )
            != 96
        ):

            return False


        if (
            m.get(
                "cluster_method"
            )
            != "KMeans"
        ):

            return False


        if (
            int(
                m.get(
                    "cluster_random_state",
                    -1,
                )
            )
            != 100
        ):

            return False


        if (
            int(
                m.get(
                    "cluster_n_init",
                    -1,
                )
            )
            != 10
        ):

            return False


        # ----------------------------------------------------
        # Arrays
        # ----------------------------------------------------

        embedding = np.load(
            required[
                "embedding"
            ]
        )

        pred = np.load(
            required[
                "pred"
            ],
            allow_pickle=True,
        )

        gt = np.load(
            required[
                "gt"
            ],
            allow_pickle=True,
        )

        coords = np.load(
            required[
                "coords"
            ]
        )

        spot_ids = np.load(
            required[
                "spot_ids"
            ],
            allow_pickle=True,
        )


        n = exp[
            "n_spots"
        ]

        K = exp["K"]


        if (
            embedding.shape
            != (n, 96)
        ):

            return False


        if (
            coords.shape
            != (n, 2)
        ):

            return False


        if (
            len(pred) != n
            or len(gt) != n
            or len(spot_ids) != n
        ):

            return False


        if not np.isfinite(
            embedding
        ).all():

            return False


        if not np.isfinite(
            coords
        ).all():

            return False


        if (
            len(
                np.unique(
                    spot_ids
                )
            )
            != n
        ):

            return False


        if (
            len(
                np.unique(
                    gt
                )
            )
            != K
        ):

            return False


        if (
            len(
                np.unique(
                    pred
                )
            )
            != K
        ):

            return False


        # ----------------------------------------------------
        # Independent metrics
        # ----------------------------------------------------

        ari = adjusted_rand_score(
            gt,
            pred,
        )

        nmi = (
            normalized_mutual_info_score(
                gt,
                pred,
                average_method="max",
            )
        )


        if not np.isclose(
            ari,
            float(
                m["ARI"]
            ),
            rtol=0,
            atol=1e-12,
        ):

            return False


        if not np.isclose(
            nmi,
            float(
                m["NMI"]
            ),
            rtol=0,
            atol=1e-12,
        ):

            return False


        return True


    except Exception:

        return False


# ============================================================
# Count existing runs
# ============================================================

existing = []


for dataset in DATASET_ORDER:

    for seed in range(10):

        if valid_existing_run(
            dataset,
            seed,
        ):

            existing.append(
                (
                    dataset,
                    seed,
                )
            )


print(
    "=" * 110
)

print(
    "MISO FORMAL 5-DATASET × 10-SEED BENCHMARK"
)

print(
    "=" * 110
)

print(
    "Existing valid formal runs:",
    len(existing),
    "/ 50"
)

print(
    "Smoke results will NOT be reused."
)

print()


# ============================================================
# Formal runs
# ============================================================

formal_start = time.time()

failed_runs = []


for dataset in DATASET_ORDER:

    for seed in range(10):


        print(
            "\n" + "=" * 110
        )

        print(
            f"FORMAL | MISO | "
            f"{dataset} | "
            f"seed={seed}"
        )

        print(
            "=" * 110
        )


        # ----------------------------------------------------
        # Resume
        # ----------------------------------------------------

        if valid_existing_run(
            dataset,
            seed,
        ):

            with formal_run_dir(
                dataset,
                seed,
            ).joinpath(
                "metrics.json"
            ).open(
                "r",
                encoding="utf-8",
            ) as f:

                old = json.load(f)


            print(
                "[SKIP] existing valid run | "
                f"ARI={old['ARI']:.6f} | "
                f"NMI={old['NMI']:.6f}"
            )

            continue


        # ----------------------------------------------------
        # Formal execution
        # ----------------------------------------------------

        try:

            result = run_miso_once(

                dataset_name=
                    dataset,

                seed=
                    seed,

                stage=
                    "formal_10seeds",
            )


            # Independent validation immediately
            assert valid_existing_run(
                dataset,
                seed,
            )


            print(
                "\nFORMAL PASS | "
                f"{dataset} | "
                f"seed={seed} | "
                f"ARI={result['ARI']:.6f} | "
                f"NMI={result['NMI']:.6f} | "
                f"K={result['predicted_K']}"
            )


        except Exception as e:

            failed_runs.append({
                "dataset":
                    dataset,

                "seed":
                    seed,

                "error":
                    repr(e),
            })


            error_dir = (
                FORMAL_ROOT
                / (
                    dataset_folder_name(
                        dataset
                    )
                    + f"_seed{seed}"
                )
            )


            error_dir.mkdir(
                parents=True,
                exist_ok=True,
            )


            (
                error_dir
                / "ERROR.txt"
            ).write_text(
                repr(e),
                encoding="utf-8",
            )


            print(
                "\nFORMAL FAIL | "
                f"{dataset} | "
                f"seed={seed}"
            )

            print(
                repr(e)
            )


# ============================================================
# Collect valid formal runs
# ============================================================

raw_rows = []


for dataset in DATASET_ORDER:

    exp = EXPECTED[
        dataset
    ]


    for seed in range(10):

        if not valid_existing_run(
            dataset,
            seed,
        ):

            continue


        run_dir = formal_run_dir(
            dataset,
            seed,
        )


        with (
            run_dir
            / "metrics.json"
        ).open(
            "r",
            encoding="utf-8",
        ) as f:

            m = json.load(f)


        raw_rows.append({

            "method":
                "MISO",

            "dataset":
                dataset,

            "training_seed":
                seed,

            "n_spots":
                exp[
                    "n_spots"
                ],

            "target_K":
                exp["K"],

            "predicted_K":
                int(
                    m[
                        "predicted_K"
                    ]
                ),

            "ARI":
                float(
                    m["ARI"]
                ),

            "NMI":
                float(
                    m["NMI"]
                ),

            "preprocessing_seconds":
                float(
                    m[
                        "preprocessing_seconds"
                    ]
                ),

            "training_seconds":
                float(
                    m[
                        "training_seconds"
                    ]
                ),

            "total_seconds":
                float(
                    m[
                        "total_seconds"
                    ]
                ),

            "RNA_features":
                int(
                    m[
                        "RNA_features_after_preprocess"
                    ]
                ),

            "second_features":
                int(
                    m[
                        "second_features_after_preprocess"
                    ]
                ),

            "embedding_dim":
                int(
                    m[
                        "embedding_dim"
                    ]
                ),
        })


raw_df = pd.DataFrame(
    raw_rows
)


RAW_PATH = (
    FORMAL_ROOT
    / "MISO_5datasets_10seeds_RAW.csv"
)


raw_df.to_csv(
    RAW_PATH,
    index=False,
)


# ============================================================
# Summary
# ============================================================

summary_rows = []


for dataset in DATASET_ORDER:

    part = raw_df[
        raw_df[
            "dataset"
        ]
        == dataset
    ]


    if len(part) == 0:

        continue


    summary_rows.append({

        "method":
            "MISO",

        "dataset":
            dataset,

        "n_runs":
            int(
                len(part)
            ),

        "ARI_mean":
            float(
                part[
                    "ARI"
                ].mean()
            ),

        "ARI_std":
            float(
                part[
                    "ARI"
                ].std(
                    ddof=0
                )
            ),

        "NMI_mean":
            float(
                part[
                    "NMI"
                ].mean()
            ),

        "NMI_std":
            float(
                part[
                    "NMI"
                ].std(
                    ddof=0
                )
            ),

        "exact_K_runs":
            int(
                (
                    part[
                        "predicted_K"
                    ]
                    ==
                    part[
                        "target_K"
                    ]
                ).sum()
            ),

        "mean_runtime_seconds":
            float(
                part[
                    "total_seconds"
                ].mean()
            ),
    })


summary_df = pd.DataFrame(
    summary_rows
)


SUMMARY_PATH = (
    FORMAL_ROOT
    / "MISO_5datasets_10seeds_SUMMARY.csv"
)


summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
)


# ============================================================
# Status report
# ============================================================

elapsed = (
    time.time()
    - formal_start
)


print(
    "\n" + "=" * 110
)

print(
    "MISO FORMAL RUN STATUS"
)

print(
    "=" * 110
)


total_valid = 0


for dataset in DATASET_ORDER:

    n_valid = sum(

        valid_existing_run(
            dataset,
            seed,
        )

        for seed in range(10)
    )


    total_valid += n_valid


    print(
        f"{dataset:8s}: "
        f"{n_valid}/10 valid runs"
    )


print(
    f"\nTOTAL: "
    f"{total_valid}/50"
)

print(
    f"Elapsed: "
    f"{elapsed / 60:.2f} min"
)

print(
    "\nRAW:",
    RAW_PATH
)


# ============================================================
# Summary display
# ============================================================

print(
    "\n" + "=" * 110
)

print(
    "MISO FORMAL SUMMARY"
)

print(
    "=" * 110
)


for _, row in (
    summary_df.iterrows()
):

    print(

        f"{row['dataset']:8s} | "

        f"ARI "
        f"{row['ARI_mean']:.6f}"
        f" ± "
        f"{row['ARI_std']:.6f}"

        f" | "

        f"NMI "
        f"{row['NMI_mean']:.6f}"
        f" ± "
        f"{row['NMI_std']:.6f}"

        f" | exact-K "
        f"{int(row['exact_K_runs'])}"
        f"/"
        f"{int(row['n_runs'])}"
    )


print(
    "\nSUMMARY:",
    SUMMARY_PATH
)


# ============================================================
# Final assertions only if all 50 complete
# ============================================================

if total_valid == 50:

    assert len(
        raw_df
    ) == 50


    assert all(

        sum(

            (
                raw_df[
                    "dataset"
                ]
                == dataset
            )

            &

            (
                raw_df[
                    "training_seed"
                ]
                == seed
            )

        )

        == 1

        for dataset
        in DATASET_ORDER

        for seed
        in range(10)
    )


    assert (
        summary_df[
            "n_runs"
        ]
        == 10
    ).all()


    assert (
        summary_df[
            "exact_K_runs"
        ]
        == 10
    ).all()


    print(
        "\nPASS: 50/50 MISO formal runs complete."
    )

    print(
        "PASS: seeds 0-9 complete for all datasets."
    )

    print(
        "PASS: all runs reached exact target K."
    )

    print(
        "PASS: std uses ddof=0."
    )


else:

    print(
        "\nWARNING: formal benchmark is incomplete."
    )

    print(
        "Re-run this Cell to resume."
    )


if len(
    failed_runs
) > 0:

    print(
        "\nFAILED RUNS:"
    )

    for item in failed_runs:

        print(
            item
        )

MISO FORMAL 5-DATASET × 10-SEED BENCHMARK
Existing valid formal runs: 0 / 50
Smoke results will NOT be reused.


FORMAL | MISO | HLN-A1 | seed=0

MISO | HLN-A1 | seed=0


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


HLN-A1: 3484 spots | K=10 | RNA + ADT
RNA features after MISO preprocess: 17954
ADT features after MISO preprocess: 31

Input spots : 3484
Target K    : 10
Modalities  : RNA + ADT
MISO sparse : False
MISO views  : all
MISO combs  : all


Training network for modality 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Training network for modality 2:   0%|          | 0/1000 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


S2-E18: 2248 spots | K=16 | RNA + ATAC
RNA features after MISO preprocess: 18709
ATAC features after MISO preprocess: 94833

Input spots : 2248
Target K    : 16
Modalities  : RNA + ATAC
MISO sparse : False
MISO views  : all
MISO combs  : all


Training network for modality 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Training network for modality 2:   0%|          | 0/1000 [00:00<?, ?it/s]


S2-E18 MISO RESULT
Embedding shape: (2248, 96)
Predicted K    : 16
ARI = 0.080762279497
NMI = 0.112744038709
Preprocessing = 5.26s
Training      = 49.77s
Total runtime = 55.27s

Saved: /kaggle/working/MISO_baseline/formal_10seeds/S2E18_seed9

PASS: MISO run completed.
PASS: 2248/2248 spots evaluated.

FORMAL PASS | S2-E18 | seed=9 | ARI=0.080762 | NMI=0.112744 | K=16

MISO FORMAL RUN STATUS
HLN-A1  : 10/10 valid runs
HLN-D1  : 10/10 valid runs
E18.5   : 10/10 valid runs
S2-E15  : 10/10 valid runs
S2-E18  : 10/10 valid runs

TOTAL: 50/50
Elapsed: 42.84 min

RAW: /kaggle/working/MISO_baseline/formal_10seeds/MISO_5datasets_10seeds_RAW.csv

MISO FORMAL SUMMARY
HLN-A1   | ARI 0.248721 ± 0.028632 | NMI 0.304995 ± 0.028080 | exact-K 10/10
HLN-D1   | ARI 0.228830 ± 0.012825 | NMI 0.280593 ± 0.014153 | exact-K 10/10
E18.5    | ARI 0.175762 ± 0.019978 | NMI 0.277499 ± 0.033038 | exact-K 10/10
S2-E15   | ARI 0.188678 ± 0.006417 | NMI 0.261746 ± 0.019929 | exact-K 10/10
S2-E18   | ARI 0.072740 ± 

Cell 10：MISO 独立审计

In [11]:
# ============================================================
# Cell 10
# MISO FINAL independent audit
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


FORMAL_ROOT = Path(
    "/kaggle/working/MISO_baseline/formal_10seeds"
)


DATASET_ORDER = [
    "HLN-A1",
    "HLN-D1",
    "E18.5",
    "S2-E15",
    "S2-E18",
]


DATASET_FOLDER = {
    "HLN-A1": "HLNA1",
    "HLN-D1": "HLND1",
    "E18.5": "E185",
    "S2-E15": "S2E15",
    "S2-E18": "S2E18",
}


EXPECTED = {
    "HLN-A1": {"n_spots": 3484, "K": 10},
    "HLN-D1": {"n_spots": 3359, "K": 11},
    "E18.5": {"n_spots": 2129, "K": 14},
    "S2-E15": {"n_spots": 1939, "K": 15},
    "S2-E18": {"n_spots": 2248, "K": 16},
}


RAW_PATH = (
    FORMAL_ROOT
    / "MISO_5datasets_10seeds_RAW.csv"
)

SUMMARY_PATH = (
    FORMAL_ROOT
    / "MISO_5datasets_10seeds_SUMMARY.csv"
)

PROTOCOL_PATH = (
    FORMAL_ROOT
    / "protocol.json"
)


assert RAW_PATH.exists()
assert SUMMARY_PATH.exists()
assert PROTOCOL_PATH.exists()


raw_saved = pd.read_csv(
    RAW_PATH
)

summary_saved = pd.read_csv(
    SUMMARY_PATH
)


audit_rows = []


print("=" * 110)
print("MISO FINAL 50-RUN INDEPENDENT AUDIT")
print("=" * 110)


for dataset in DATASET_ORDER:

    n = EXPECTED[dataset]["n_spots"]
    K = EXPECTED[dataset]["K"]

    print(f"\nAuditing {dataset}...")


    for seed in range(10):

        run_dir = (
            FORMAL_ROOT
            / f"{DATASET_FOLDER[dataset]}_seed{seed}"
        )


        paths = {
            "metrics":
                run_dir / "metrics.json",

            "embedding":
                run_dir / "embedding.npy",

            "pred":
                run_dir / "pred_labels.npy",

            "gt":
                run_dir / "gt_labels.npy",

            "coords":
                run_dir / "coords.npy",

            "spot_ids":
                run_dir / "spot_ids.npy",
        }


        for name, path in paths.items():

            assert path.exists(), (
                f"{dataset} seed={seed}: "
                f"missing {name}"
            )


        with paths["metrics"].open(
            "r",
            encoding="utf-8",
        ) as f:

            m = json.load(f)


        embedding = np.load(
            paths["embedding"]
        )

        pred = np.load(
            paths["pred"],
            allow_pickle=True,
        )

        gt = np.load(
            paths["gt"],
            allow_pickle=True,
        )

        coords = np.load(
            paths["coords"]
        )

        spot_ids = np.load(
            paths["spot_ids"],
            allow_pickle=True,
        )


        # ---------------------------------------------
        # Metadata
        # ---------------------------------------------

        assert m["method"] == "MISO"
        assert m["dataset"] == dataset

        assert (
            int(m["training_seed"])
            == seed
        )

        assert (
            int(m["n_spots"])
            == n
        )

        assert (
            int(m["target_K"])
            == K
        )

        assert (
            int(m["predicted_K"])
            == K
        )

        assert (
            int(m["embedding_dim"])
            == 96
        )

        assert (
            m["cluster_method"]
            == "KMeans"
        )

        assert (
            int(
                m["cluster_random_state"]
            )
            == 100
        )

        assert (
            int(
                m["cluster_n_init"]
            )
            == 10
        )


        # ---------------------------------------------
        # Shapes
        # ---------------------------------------------

        assert (
            embedding.shape
            == (n, 96)
        )

        assert (
            coords.shape
            == (n, 2)
        )

        assert len(pred) == n
        assert len(gt) == n
        assert len(spot_ids) == n


        assert np.isfinite(
            embedding
        ).all()

        assert np.isfinite(
            coords
        ).all()


        assert (
            len(
                np.unique(
                    spot_ids
                )
            )
            == n
        )


        # ---------------------------------------------
        # K
        # ---------------------------------------------

        assert (
            len(
                np.unique(gt)
            )
            == K
        )

        assert (
            len(
                np.unique(pred)
            )
            == K
        )


        # ---------------------------------------------
        # Independent metrics
        # ---------------------------------------------

        ari = adjusted_rand_score(
            gt,
            pred,
        )


        nmi = (
            normalized_mutual_info_score(
                gt,
                pred,
                average_method="max",
            )
        )


        assert np.isclose(
            ari,
            float(m["ARI"]),
            rtol=0,
            atol=1e-12,
        )


        assert np.isclose(
            nmi,
            float(m["NMI"]),
            rtol=0,
            atol=1e-12,
        )


        audit_rows.append({

            "dataset":
                dataset,

            "training_seed":
                seed,

            "n_spots":
                n,

            "target_K":
                K,

            "predicted_K":
                len(
                    np.unique(pred)
                ),

            "ARI":
                float(ari),

            "NMI":
                float(nmi),
        })


    print(
        f"{dataset}: 10/10 PASS"
    )


# ============================================================
# RAW audit
# ============================================================

audit_raw = pd.DataFrame(
    audit_rows
)

assert len(audit_raw) == 50


a = (
    audit_raw[
        [
            "dataset",
            "training_seed",
            "ARI",
            "NMI",
        ]
    ]
    .sort_values(
        [
            "dataset",
            "training_seed",
        ]
    )
    .reset_index(drop=True)
)


b = (
    raw_saved[
        [
            "dataset",
            "training_seed",
            "ARI",
            "NMI",
        ]
    ]
    .sort_values(
        [
            "dataset",
            "training_seed",
        ]
    )
    .reset_index(drop=True)
)


assert a[
    ["dataset", "training_seed"]
].equals(
    b[
        ["dataset", "training_seed"]
    ]
)


assert np.allclose(
    a["ARI"],
    b["ARI"],
    rtol=0,
    atol=1e-12,
)


assert np.allclose(
    a["NMI"],
    b["NMI"],
    rtol=0,
    atol=1e-12,
)


print(
    "\nPASS: RAW.csv verified."
)


# ============================================================
# SUMMARY audit
# ============================================================

summary_rows = []


for dataset in DATASET_ORDER:

    part = audit_raw[
        audit_raw["dataset"]
        == dataset
    ]


    summary_rows.append({

        "dataset":
            dataset,

        "n_runs":
            10,

        "ARI_mean":
            float(
                part["ARI"].mean()
            ),

        "ARI_std":
            float(
                part["ARI"].std(
                    ddof=0
                )
            ),

        "NMI_mean":
            float(
                part["NMI"].mean()
            ),

        "NMI_std":
            float(
                part["NMI"].std(
                    ddof=0
                )
            ),

        "exact_K_runs":
            int(
                (
                    part["predicted_K"]
                    ==
                    part["target_K"]
                ).sum()
            ),
    })


audit_summary = pd.DataFrame(
    summary_rows
)


saved_summary = (
    summary_saved
    .set_index("dataset")
    .loc[DATASET_ORDER]
    .reset_index()
)


for col in [
    "ARI_mean",
    "ARI_std",
    "NMI_mean",
    "NMI_std",
]:

    assert np.allclose(
        audit_summary[col],
        saved_summary[col],
        rtol=0,
        atol=1e-12,
    )


assert (
    audit_summary[
        "exact_K_runs"
    ]
    == 10
).all()


print(
    "PASS: SUMMARY.csv verified."
)


# ============================================================
# Save audit
# ============================================================

AUDIT_RAW = (
    FORMAL_ROOT
    / "MISO_5datasets_10seeds_AUDIT_RAW.csv"
)

AUDIT_SUMMARY = (
    FORMAL_ROOT
    / "MISO_5datasets_10seeds_AUDIT_SUMMARY.csv"
)


audit_raw.to_csv(
    AUDIT_RAW,
    index=False,
)

audit_summary.to_csv(
    AUDIT_SUMMARY,
    index=False,
)


print(
    "\n" + "=" * 110
)

print(
    "MISO INDEPENDENT SUMMARY"
)

print(
    "=" * 110
)


for _, row in audit_summary.iterrows():

    print(
        f"{row['dataset']:8s} | "
        f"ARI "
        f"{row['ARI_mean']:.6f}"
        f" ± "
        f"{row['ARI_std']:.6f}"
        f" | NMI "
        f"{row['NMI_mean']:.6f}"
        f" ± "
        f"{row['NMI_std']:.6f}"
        f" | exact-K "
        f"{int(row['exact_K_runs'])}/10"
    )


print(
    "\nPASS: 50/50 MISO runs independently verified."
)

print(
    "PASS: metrics.json == recomputed metrics."
)

print(
    "PASS: RAW == independent recomputation."
)

print(
    "PASS: SUMMARY == independent recomputation."
)

print(
    "PASS: embedding shape = N × 96 for all runs."
)

print(
    "PASS: all 50 runs have exact target K."
)

print(
    "PASS: std uses ddof=0."
)

MISO FINAL 50-RUN INDEPENDENT AUDIT

Auditing HLN-A1...
HLN-A1: 10/10 PASS

Auditing HLN-D1...
HLN-D1: 10/10 PASS

Auditing E18.5...
E18.5: 10/10 PASS

Auditing S2-E15...
S2-E15: 10/10 PASS

Auditing S2-E18...
S2-E18: 10/10 PASS

PASS: RAW.csv verified.
PASS: SUMMARY.csv verified.

MISO INDEPENDENT SUMMARY
HLN-A1   | ARI 0.248721 ± 0.028632 | NMI 0.304995 ± 0.028080 | exact-K 10/10
HLN-D1   | ARI 0.228830 ± 0.012825 | NMI 0.280593 ± 0.014153 | exact-K 10/10
E18.5    | ARI 0.175762 ± 0.019978 | NMI 0.277499 ± 0.033038 | exact-K 10/10
S2-E15   | ARI 0.188678 ± 0.006417 | NMI 0.261746 ± 0.019929 | exact-K 10/10
S2-E18   | ARI 0.072740 ± 0.010165 | NMI 0.108509 ± 0.008849 | exact-K 10/10

PASS: 50/50 MISO runs independently verified.
PASS: metrics.json == recomputed metrics.
PASS: RAW == independent recomputation.
PASS: SUMMARY == independent recomputation.
PASS: embedding shape = N × 96 for all runs.
PASS: all 50 runs have exact target K.
PASS: std uses ddof=0.


Cell 10.5A：检查 S2-E18 输入是不是和另外两个 ATAC 数据集的数据形态不同

In [12]:
# ============================================================
# Cell 10.5A
# MISO ATAC input representation audit
#
# Diagnostic only. Does NOT modify formal results.
# ============================================================

import numpy as np
import scipy.sparse as sp
import scanpy as sc


def matrix_stats(X, name):

    if sp.issparse(X):

        values = X.data
        nnz = X.nnz
        total = X.shape[0] * X.shape[1]

    else:

        arr = np.asarray(X)

        values = arr[
            arr != 0
        ]

        nnz = len(values)
        total = arr.size


    if len(values) > 100000:

        rng = np.random.default_rng(0)

        idx = rng.choice(
            len(values),
            size=100000,
            replace=False,
        )

        sample = values[idx]

    else:

        sample = values


    sample = np.asarray(
        sample,
        dtype=np.float64,
    )


    integer_like = np.mean(
        np.isclose(
            sample,
            np.round(sample),
            atol=1e-6,
        )
    )


    print("\n", name)
    print("-" * 80)

    print(
        "shape              :",
        X.shape
    )

    print(
        "sparse             :",
        sp.issparse(X)
    )

    print(
        "nonzero fraction   :",
        nnz / total
    )

    print(
        "nonzero min        :",
        sample.min()
    )

    print(
        "nonzero median     :",
        np.median(sample)
    )

    print(
        "nonzero mean       :",
        sample.mean()
    )

    print(
        "nonzero max        :",
        sample.max()
    )

    print(
        "integer-like frac  :",
        integer_like
    )

    print(
        "negative frac      :",
        np.mean(sample < 0)
    )


for dataset in [
    "E18.5",
    "S2-E15",
    "S2-E18",
]:

    print(
        "\n" + "=" * 100
    )

    print(
        dataset
    )

    print(
        "=" * 100
    )


    spec = DATASET_SPECS[
        dataset
    ]


    rna = sc.read_h5ad(
        spec["rna"]
    )

    atac = sc.read_h5ad(
        spec["second"]
    )


    matrix_stats(
        rna.X,
        "RNA X before MISO preprocess"
    )

    matrix_stats(
        atac.X,
        "ATAC X before MISO preprocess"
    )


    print(
        "\nRNA layers:",
        list(rna.layers.keys())
    )

    print(
        "ATAC layers:",
        list(atac.layers.keys())
    )

    print(
        "RNA .raw exists:",
        rna.raw is not None
    )

    print(
        "ATAC .raw exists:",
        atac.raw is not None
    )


E18.5

 RNA X before MISO preprocess
--------------------------------------------------------------------------------
shape              : (2129, 32285)
sparse             : False
nonzero fraction   : 0.06566209981222748
nonzero min        : 1.0
nonzero median     : 1.0
nonzero mean       : 1.64595
nonzero max        : 1263.0
integer-like frac  : 1.0
negative frac      : 0.0

 ATAC X before MISO preprocess
--------------------------------------------------------------------------------
shape              : (2129, 161461)
sparse             : True
nonzero fraction   : 0.051626792107736734
nonzero min        : 1.0
nonzero median     : 1.0
nonzero mean       : 1.40241
nonzero max        : 751.0
integer-like frac  : 1.0
negative frac      : 0.0

RNA layers: []
ATAC layers: []
RNA .raw exists: False
ATAC .raw exists: False

S2-E15


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



 RNA X before MISO preprocess
--------------------------------------------------------------------------------
shape              : (1939, 32285)
sparse             : True
nonzero fraction   : 0.10567899372873574
nonzero min        : 1.0
nonzero median     : 1.0
nonzero mean       : 1.88228
nonzero max        : 419.0
integer-like frac  : 1.0
negative frac      : 0.0

 ATAC X before MISO preprocess
--------------------------------------------------------------------------------
shape              : (1939, 100329)
sparse             : True
nonzero fraction   : 0.060738072720635645
nonzero min        : 1.0
nonzero median     : 2.0
nonzero mean       : 2.49665
nonzero max        : 354.0
integer-like frac  : 1.0
negative frac      : 0.0

RNA layers: []
ATAC layers: []
RNA .raw exists: False
ATAC .raw exists: False

S2-E18


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



 RNA X before MISO preprocess
--------------------------------------------------------------------------------
shape              : (2248, 32285)
sparse             : True
nonzero fraction   : 0.08305082293651349
nonzero min        : 1.0
nonzero median     : 1.0
nonzero mean       : 1.9121
nonzero max        : 1797.0
integer-like frac  : 1.0
negative frac      : 0.0

 ATAC X before MISO preprocess
--------------------------------------------------------------------------------
shape              : (2248, 94941)
sparse             : True
nonzero fraction   : 0.05207779632085422
nonzero min        : 1.0
nonzero median     : 2.0
nonzero mean       : 2.42727
nonzero max        : 242.0
integer-like frac  : 1.0
negative frac      : 0.0

RNA layers: []
ATAC layers: []
RNA .raw exists: False
ATAC .raw exists: False


Cell 10.5B：看看到底是 RNA、ATAC 还是 interaction 把结果拉低了

In [13]:
# ============================================================
# Cell 10.5B
# Diagnose individual MISO embedding components
#
# GT is used for diagnosis only.
# NOT part of formal evaluation / model selection.
# ============================================================

import json
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


DATASETS_DIAG = [
    "E18.5",
    "S2-E15",
    "S2-E18",
]


COMPONENTS = {

    "RNA":
        slice(0, 32),

    "Second":
        slice(32, 64),

    "Interaction":
        slice(64, 96),

    "Full":
        slice(0, 96),
}


rows = []


for dataset in DATASETS_DIAG:

    K = EXPECTED[
        dataset
    ]["K"]


    for seed in range(10):

        run_dir = formal_run_dir(
            dataset,
            seed,
        )


        emb = np.load(
            run_dir
            / "embedding.npy"
        )

        gt = np.load(
            run_dir
            / "gt_labels.npy",
            allow_pickle=True,
        )


        for component, sl in COMPONENTS.items():

            Z = emb[:, sl]


            pred = KMeans(
                n_clusters=K,
                random_state=100,
                n_init=10,
            ).fit_predict(Z)


            ari = adjusted_rand_score(
                gt,
                pred,
            )

            nmi = (
                normalized_mutual_info_score(
                    gt,
                    pred,
                    average_method="max",
                )
            )


            rows.append({

                "dataset":
                    dataset,

                "seed":
                    seed,

                "component":
                    component,

                "ARI":
                    ari,

                "NMI":
                    nmi,
            })


diag = pd.DataFrame(rows)


summary = (

    diag.groupby(
        [
            "dataset",
            "component",
        ],
        sort=False,
    )

    .agg(
        ARI_mean=("ARI", "mean"),
        ARI_std=("ARI", lambda x: np.std(x, ddof=0)),
        NMI_mean=("NMI", "mean"),
        NMI_std=("NMI", lambda x: np.std(x, ddof=0)),
    )

    .reset_index()
)


print(
    summary.to_string(
        index=False
    )
)

dataset   component  ARI_mean  ARI_std  NMI_mean  NMI_std
  E18.5         RNA  0.072431 0.008904  0.126586 0.015782
  E18.5      Second  0.193262 0.006394  0.343533 0.013375
  E18.5 Interaction  0.012422 0.025565  0.015316 0.015264
  E18.5        Full  0.175762 0.019978  0.277499 0.033038
 S2-E15         RNA  0.174287 0.018995  0.280992 0.016391
 S2-E15      Second  0.163567 0.006161  0.269391 0.015813
 S2-E15 Interaction  0.002803 0.000222  0.009653 0.000220
 S2-E15        Full  0.188678 0.006417  0.261746 0.019929
 S2-E18         RNA  0.085511 0.002363  0.115587 0.007664
 S2-E18      Second  0.076785 0.012638  0.128622 0.028441
 S2-E18 Interaction  0.002766 0.000475  0.009546 0.000416
 S2-E18        Full  0.072740 0.010165  0.108509 0.008849


Cell 10.5C：额外跑一次官方 tutorial 的 seed=100

In [14]:
# ============================================================
# Cell 10.5C
# Official-tutorial seed=100 diagnostic
#
# NOT part of formal results.
# ============================================================

S2E18_SEED100_DIAG = run_miso_once(

    dataset_name="S2-E18",

    seed=100,

    stage="diagnostic_seed100",
)


print(
    "\n" + "=" * 100
)

print(
    "S2-E18 OFFICIAL-SEED DIAGNOSTIC"
)

print(
    "=" * 100
)

print(
    "ARI:",
    S2E18_SEED100_DIAG["ARI"]
)

print(
    "NMI:",
    S2E18_SEED100_DIAG["NMI"]
)


MISO | S2-E18 | seed=100


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


S2-E18: 2248 spots | K=16 | RNA + ATAC
RNA features after MISO preprocess: 18709
ATAC features after MISO preprocess: 94833

Input spots : 2248
Target K    : 16
Modalities  : RNA + ATAC
MISO sparse : False
MISO views  : all
MISO combs  : all


Training network for modality 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Training network for modality 2:   0%|          | 0/1000 [00:00<?, ?it/s]


S2-E18 MISO RESULT
Embedding shape: (2248, 96)
Predicted K    : 16
ARI = 0.078321464687
NMI = 0.108801222508
Preprocessing = 5.48s
Training      = 50.26s
Total runtime = 55.99s

Saved: /kaggle/working/MISO_baseline/diagnostic_seed100/S2E18_seed100

PASS: MISO run completed.
PASS: 2248/2248 spots evaluated.

S2-E18 OFFICIAL-SEED DIAGNOSTIC
ARI: 0.07832146468729022
NMI: 0.10880122250756888


OldEnv Cell 1R2：接受 ToS + 创建 Python 3.7.13

In [17]:
# ============================================================
# OldEnv Cell 1R2
# Create exact Python 3.7.13 environment from Anaconda main
# ============================================================

from pathlib import Path
import shutil
import subprocess
import os


WORK = Path("/kaggle/working")

CONDA_ROOT = (
    WORK / "miniconda_miso"
)

ENV_ROOT = (
    WORK / "miso_py37"
)

CONDA = (
    CONDA_ROOT / "bin/conda"
)

PY37 = (
    ENV_ROOT / "bin/python"
)


assert CONDA.exists(), CONDA


print("=" * 100)
print("MISO OLD ENV — PYTHON 3.7.13")
print("=" * 100)

print("Conda:", CONDA)


# ============================================================
# Clean subprocess environment
#
# Prevent Kaggle Python 3.12 PYTHONPATH from leaking into
# the isolated Python 3.7 environment.
# ============================================================

clean_env = os.environ.copy()

clean_env.pop(
    "PYTHONPATH",
    None,
)


# ============================================================
# Accept ToS ONLY for this private Miniconda installation
# ============================================================

for channel in [
    "https://repo.anaconda.com/pkgs/main",
    "https://repo.anaconda.com/pkgs/r",
]:

    print(
        "\nAccepting ToS:",
        channel
    )

    subprocess.run(
        [
            str(CONDA),
            "tos",
            "accept",
            "--override-channels",
            "--channel",
            channel,
        ],
        check=True,
        env=clean_env,
    )


print(
    "\nPASS: Anaconda channel ToS accepted."
)


# ============================================================
# Remove incomplete previous environment
# ============================================================

if ENV_ROOT.exists():

    if not PY37.exists():

        print(
            "\nRemoving incomplete environment:",
            ENV_ROOT
        )

        shutil.rmtree(
            ENV_ROOT
        )


# ============================================================
# Create exact official-style Python environment
#
# python 3.7.13 is available in Anaconda main.
# ============================================================

if not PY37.exists():

    print(
        "\nCreating Python 3.7.13 environment..."
    )

    subprocess.run(
        [
            str(CONDA),
            "create",
            "-y",

            "-p",
            str(ENV_ROOT),

            "--override-channels",

            "-c",
            "https://repo.anaconda.com/pkgs/main",

            "-c",
            "https://repo.anaconda.com/pkgs/r",

            "python=3.7.13",
            "pip=22.3.1",
            "setuptools=65.6.3",
            "wheel",
        ],
        check=True,
        env=clean_env,
    )


assert PY37.exists(), PY37


# ============================================================
# Verify
# ============================================================

print(
    "\nPython executable:",
    PY37
)


subprocess.run(
    [
        str(PY37),
        "--version",
    ],
    check=True,
    env=clean_env,
)


verify_code = r'''
import os
import sys

print("Python version :", sys.version)
print("Executable     :", sys.executable)
print("PYTHONPATH     :", os.environ.get("PYTHONPATH"))
'''


subprocess.run(
    [
        str(PY37),
        "-c",
        verify_code,
    ],
    check=True,
    env=clean_env,
)


print(
    "\nPASS: exact Python 3.7.13 environment ready."
)

print(
    "PASS: Kaggle Python 3.12 PYTHONPATH isolated."
)

MISO OLD ENV — PYTHON 3.7.13
Conda: /kaggle/working/miniconda_miso/bin/conda

Accepting ToS: https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/main

Accepting ToS: https://repo.anaconda.com/pkgs/r
accepted Terms of Service for https://repo.anaconda.com/pkgs/r

PASS: Anaconda channel ToS accepted.

Creating Python 3.7.13 environment...
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: ...working... done
Channels:
 - defaults
Platform: linux-64
Solving environment: ...working... done




==> WARNING: A newer version of conda exists. <==
    current version: 26.7.1
    latest version: 26.7.2

Please update conda by running

    $ conda self update





## Package Plan ##

  environment location: /kaggle/working/miso_py37

  added / updated specs:
    - pip=22.3.1
    - python=3.7.13
    - setuptools=65.6.3
    - wheel


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    certifi-2022.12.7          |   py37h06a4308_0         150 KB
    libffi-3.3                 |       he6710b0_2          50 KB
    openssl-1.1.1w             |       h7f8727e_0         3.7 MB
    pip-22.3.1                 |   py37h06a4308_0         2.7 MB
    python-3.7.13              |       haa1d7c7_1        40.7 MB
    setuptools-65.6.3          |   py37h06a4308_0         1.1 MB
    sqlite-3.53.4              |       h795bf6d_0         1.2 MB
    tk-8.6.15                  |       h54e0aa7_0         3.4 MB
    wheel-0.38.4               |   py37h06a4308_0          63 KB
    ------------------------------------------------------------
                                  

WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/conda/en/stable/new-features.html




Python executable: /kaggle/working/miso_py37/bin/python
Python 3.7.13
Python version : 3.7.13 (default, Oct 18 2022, 18:57:03) 
[GCC 11.2.0]
Executable     : /kaggle/working/miso_py37/bin/python
PYTHONPATH     : None

PASS: exact Python 3.7.13 environment ready.
PASS: Kaggle Python 3.12 PYTHONPATH isolated.


OldEnv Cell 2R：安装 MISO 官方旧依赖

In [18]:
# ============================================================
# OldEnv Cell 2R
# Install official MISO + pinned dependencies
# inside isolated Python 3.7.13
# ============================================================

from pathlib import Path
import subprocess
import os


PY37 = Path(
    "/kaggle/working/miso_py37/bin/python"
)

MISO_ROOT = Path(
    "/kaggle/working/miso"
)


assert PY37.exists(), PY37
assert MISO_ROOT.exists(), MISO_ROOT


clean_env = os.environ.copy()

clean_env.pop(
    "PYTHONPATH",
    None,
)

clean_env[
    "PIP_DISABLE_PIP_VERSION_CHECK"
] = "1"


print("=" * 100)
print("INSTALL OFFICIAL MISO OLD ENV")
print("=" * 100)

print(
    "Python:",
    PY37
)

print(
    "MISO repo:",
    MISO_ROOT
)


# ------------------------------------------------------------
# Packaging tools
# ------------------------------------------------------------

subprocess.run(
    [
        str(PY37),
        "-m",
        "pip",
        "install",
        "--no-cache-dir",

        "pip==22.3.1",
        "setuptools==65.6.3",
        "wheel==0.38.4",
    ],
    check=True,
    env=clean_env,
)


# ------------------------------------------------------------
# Install official MISO package.
#
# Its setup.py supplies the historical pinned dependencies.
# ------------------------------------------------------------

subprocess.run(
    [
        str(PY37),
        "-m",
        "pip",
        "install",
        "--no-cache-dir",

        str(MISO_ROOT),
    ],
    check=True,
    env=clean_env,
)


print(
    "\nPASS: official MISO installed in Python 3.7.13 env."
)

INSTALL OFFICIAL MISO OLD ENV
Python: /kaggle/working/miso_py37/bin/python
MISO repo: /kaggle/working/miso
Processing ./miso
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.8/24.8 MB 207.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 207.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.5/887.5 MB 259.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 238.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 218.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 245.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 MB 227.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 151.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 172.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

OldEnv Cell 3：环境审计

In [19]:
# ============================================================
# OldEnv Cell 3
# Exact old-environment audit
# ============================================================

from pathlib import Path
import subprocess
import os


PY37 = Path(
    "/kaggle/working/miso_py37/bin/python"
)


clean_env = os.environ.copy()

clean_env.pop(
    "PYTHONPATH",
    None,
)


audit_code = r'''
import platform

import numpy
import scipy
import sklearn
import scanpy
import anndata
import torch

from miso import Miso


print("=" * 90)
print("MISO OLD-ENV AUDIT")
print("=" * 90)

print("Python        :", platform.python_version())
print("NumPy         :", numpy.__version__)
print("SciPy         :", scipy.__version__)
print("sklearn       :", sklearn.__version__)
print("Scanpy        :", scanpy.__version__)
print("AnnData       :", anndata.__version__)
print("PyTorch       :", torch.__version__)
print("CUDA runtime  :", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU           :",
        torch.cuda.get_device_name(0)
    )

print()
print("MISO import   : PASS")
'''


subprocess.run(
    [
        str(PY37),
        "-c",
        audit_code,
    ],
    check=True,
    env=clean_env,
)

MISO OLD-ENV AUDIT
Python        : 3.7.13
NumPy         : 1.21.6
SciPy         : 1.7.3
sklearn       : 1.0.2
Scanpy        : 1.9.1
AnnData       : 0.8.0
PyTorch       : 1.13.1+cu117
CUDA runtime  : 11.7
CUDA available: True
GPU           : Tesla T4

MISO import   : PASS


CompletedProcess(args=['/kaggle/working/miso_py37/bin/python', '-c', '\nimport platform\n\nimport numpy\nimport scipy\nimport sklearn\nimport scanpy\nimport anndata\nimport torch\n\nfrom miso import Miso\n\n\nprint("=" * 90)\nprint("MISO OLD-ENV AUDIT")\nprint("=" * 90)\n\nprint("Python        :", platform.python_version())\nprint("NumPy         :", numpy.__version__)\nprint("SciPy         :", scipy.__version__)\nprint("sklearn       :", sklearn.__version__)\nprint("Scanpy        :", scanpy.__version__)\nprint("AnnData       :", anndata.__version__)\nprint("PyTorch       :", torch.__version__)\nprint("CUDA runtime  :", torch.version.cuda)\nprint("CUDA available:", torch.cuda.is_available())\n\nif torch.cuda.is_available():\n    print(\n        "GPU           :",\n        torch.cuda.get_device_name(0)\n    )\n\nprint()\nprint("MISO import   : PASS")\n'], returncode=0)

OldEnv Cell 4：写诊断脚本

In [20]:
# ============================================================
# OldEnv Cell 4
# MISO official-old-environment diagnostic
# S2-E18 | seeds = 0,1,2,100
#
# Diagnostic only.
# Does NOT overwrite current formal MISO results.
# ============================================================

from pathlib import Path


SCRIPT = Path(
    "/kaggle/working/run_miso_oldenv_s2e18.py"
)


code = r'''
import os
import sys
import gc
import json
import time
import random
from pathlib import Path

import numpy as np
import scanpy as sc
import torch

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)

from miso import Miso
from miso.utils import preprocess


# ============================================================
# Paths
# ============================================================

DATA_ROOT = Path(
    "/kaggle/input/datasets/wuvdji/smgc-data"
)

RNA_PATH = (
    DATA_ROOT
    / "Mouse_Embryos_S2/E18/adata_RNA.h5ad"
)

ATAC_PATH = (
    DATA_ROOT
    / "Mouse_Embryos_S2/E18/adata_ATAC.h5ad"
)

OUT_ROOT = Path(
    "/kaggle/working/MISO_oldenv_S2E18_diag"
)

OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


EXPECTED_N = 2248
TARGET_K = 16

SEEDS = [
    0,
    1,
    2,
    100,
]


# ============================================================
# Seed
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# Run
# ============================================================

results = []


for seed in SEEDS:

    print(
        "\n" + "=" * 110
    )

    print(
        "MISO OLD ENV | "
        "S2-E18 | "
        "seed={}".format(seed)
    )

    print(
        "=" * 110
    )


    start = time.time()


    # --------------------------------------------------------
    # Load benchmark data
    # --------------------------------------------------------

    rna = sc.read_h5ad(
        RNA_PATH
    )

    atac = sc.read_h5ad(
        ATAC_PATH
    )


    assert rna.n_obs == EXPECTED_N
    assert atac.n_obs == EXPECTED_N


    assert np.array_equal(
        np.asarray(rna.obs_names),
        np.asarray(atac.obs_names),
    )


    gt = (
        rna.obs["Spatial_Label"]
        .astype(str)
        .to_numpy()
    )


    assert len(gt) == EXPECTED_N

    assert (
        len(np.unique(gt))
        == TARGET_K
    )


    # --------------------------------------------------------
    # Official MISO preprocessing
    # --------------------------------------------------------

    X_rna = preprocess(
        rna.copy(),
        modality="rna",
    )

    X_atac = preprocess(
        atac.copy(),
        modality="atac",
    )


    X_rna = np.asarray(
        X_rna,
        dtype=np.float32,
    )

    X_atac = np.asarray(
        X_atac,
        dtype=np.float32,
    )


    assert X_rna.shape[0] == EXPECTED_N
    assert X_atac.shape[0] == EXPECTED_N

    assert np.isfinite(X_rna).all()
    assert np.isfinite(X_atac).all()


    print(
        "RNA after preprocess :",
        X_rna.shape
    )

    print(
        "ATAC after preprocess:",
        X_atac.shape
    )


    # --------------------------------------------------------
    # Seed
    # --------------------------------------------------------

    set_seed(seed)


    device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )


    print(
        "Device:",
        device
    )


    # --------------------------------------------------------
    # Official MISO model
    # --------------------------------------------------------

    model = Miso(
        [
            X_rna,
            X_atac,
        ],
        ind_views="all",
        combs="all",
        sparse=False,
        device=device,
    )


    model.train()


    embedding = np.asarray(
        model.emb
    )


    assert (
        embedding.shape
        == (
            EXPECTED_N,
            96,
        )
    )

    assert np.isfinite(
        embedding
    ).all()


    # --------------------------------------------------------
    # IMPORTANT:
    # use official model.cluster()
    #
    # sklearn==1.0.2 here,
    # therefore official historical default n_init applies.
    # --------------------------------------------------------

    pred = model.cluster(
        n_clusters=TARGET_K
    )


    pred = np.asarray(
        pred
    )


    assert len(pred) == EXPECTED_N

    assert (
        len(np.unique(pred))
        == TARGET_K
    )


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    ari = adjusted_rand_score(
        gt,
        pred,
    )

    nmi = (
        normalized_mutual_info_score(
            gt,
            pred,
            average_method="max",
        )
    )


    runtime = (
        time.time()
        - start
    )


    print(
        "\nEmbedding shape:",
        embedding.shape
    )

    print(
        "Predicted K    :",
        len(np.unique(pred))
    )

    print(
        "ARI = {:.12f}".format(
            ari
        )
    )

    print(
        "NMI = {:.12f}".format(
            nmi
        )
    )

    print(
        "Runtime = {:.2f}s".format(
            runtime
        )
    )


    # --------------------------------------------------------
    # Save evidence
    # --------------------------------------------------------

    run_dir = (
        OUT_ROOT
        / "seed{}".format(seed)
    )

    run_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    np.save(
        run_dir / "embedding.npy",
        embedding,
    )

    np.save(
        run_dir / "pred_labels.npy",
        pred,
    )

    np.save(
        run_dir / "gt_labels.npy",
        gt,
    )


    row = {

        "dataset":
            "S2-E18",

        "seed":
            int(seed),

        "n_spots":
            EXPECTED_N,

        "target_K":
            TARGET_K,

        "predicted_K":
            int(
                len(np.unique(pred))
            ),

        "RNA_features":
            int(X_rna.shape[1]),

        "ATAC_features":
            int(X_atac.shape[1]),

        "embedding_dim":
            int(embedding.shape[1]),

        "ARI":
            float(ari),

        "NMI":
            float(nmi),

        "runtime_seconds":
            float(runtime),

        "python":
            sys.version.split()[0],

        "torch":
            torch.__version__,
    }


    with (
        run_dir
        / "metrics.json"
    ).open(
        "w"
    ) as f:

        json.dump(
            row,
            f,
            indent=2,
        )


    results.append(
        row
    )


    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    del model
    del rna
    del atac
    del X_rna
    del X_atac
    del embedding

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# Summary
# ============================================================

aris = np.asarray(
    [
        x["ARI"]
        for x in results
    ],
    dtype=float,
)

nmis = np.asarray(
    [
        x["NMI"]
        for x in results
    ],
    dtype=float,
)


print(
    "\n" + "=" * 110
)

print(
    "MISO OLD-ENV S2-E18 SUMMARY"
)

print(
    "=" * 110
)


for x in results:

    print(
        "seed={:3d} | "
        "ARI {:.6f} | "
        "NMI {:.6f} | "
        "runtime {:.2f}s".format(
            x["seed"],
            x["ARI"],
            x["NMI"],
            x["runtime_seconds"],
        )
    )


print()

print(
    "ARI mean ± std = "
    "{:.6f} ± {:.6f}".format(
        aris.mean(),
        aris.std(ddof=0),
    )
)

print(
    "NMI mean ± std = "
    "{:.6f} ± {:.6f}".format(
        nmis.mean(),
        nmis.std(ddof=0),
    )
)


summary = {

    "seeds":
        SEEDS,

    "ARI_mean":
        float(
            aris.mean()
        ),

    "ARI_std":
        float(
            aris.std(ddof=0)
        ),

    "NMI_mean":
        float(
            nmis.mean()
        ),

    "NMI_std":
        float(
            nmis.std(ddof=0)
        ),
}


with (
    OUT_ROOT
    / "summary.json"
).open(
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
    )


print(
    "\nSaved:",
    OUT_ROOT
)

print(
    "\nPASS: old-environment diagnostic complete."
)
'''


SCRIPT.write_text(
    code,
    encoding="utf-8",
)


print(
    "Script written:"
)

print(
    SCRIPT
)

Script written:
/kaggle/working/run_miso_oldenv_s2e18.py


OldEnv Cell 5：运行

In [21]:
# ============================================================
# OldEnv Cell 5
# Execute diagnostic using exact Python 3.7 environment
# ============================================================

from pathlib import Path
import subprocess
import os


PY37 = Path(
    "/kaggle/working/miso_py37/bin/python"
)

SCRIPT = Path(
    "/kaggle/working/run_miso_oldenv_s2e18.py"
)


assert PY37.exists()
assert SCRIPT.exists()


clean_env = os.environ.copy()

clean_env.pop(
    "PYTHONPATH",
    None,
)


# Make CUDA deterministic where supported
clean_env[
    "CUBLAS_WORKSPACE_CONFIG"
] = ":4096:8"


print("=" * 100)
print("RUN MISO OLD-ENV S2-E18 DIAGNOSTIC")
print("=" * 100)


result = subprocess.run(
    [
        str(PY37),
        "-u",
        str(SCRIPT),
    ],
    check=True,
    env=clean_env,
)


print(
    "\nSubprocess return code:",
    result.returncode
)

RUN MISO OLD-ENV S2-E18 DIAGNOSTIC

MISO OLD ENV | S2-E18 | seed=0


/kaggle/working/miso_py37/lib/python3.7/site-packages/anndata/_core/anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


RNA after preprocess : (2248, 18709)
ATAC after preprocess: (2248, 94833)
Device: cuda


Training network for modality 2: 100%|██████████| 1000/1000 [00:08<00:00, 123.17it/s]



Embedding shape: (2248, 96)
Predicted K    : 16
ARI = 0.085676652622
NMI = 0.124608209304
Runtime = 61.00s

MISO OLD ENV | S2-E18 | seed=1


...


Embedding shape: (2248, 96)
Predicted K    : 16
ARI = 0.063738827404
NMI = 0.101724995899
Runtime = 59.26s

MISO OLD ENV | S2-E18 | seed=100


/kaggle/working/miso_py37/lib/python3.7/site-packages/anndata/_core/anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


RNA after preprocess : (2248, 18709)
ATAC after preprocess: (2248, 94833)
Device: cuda


Training network for modality 2: 100%|██████████| 1000/1000 [00:08<00:00, 122.85it/s]



Embedding shape: (2248, 96)
Predicted K    : 16
ARI = 0.075202092465
NMI = 0.107189852459
Runtime = 59.23s

MISO OLD-ENV S2-E18 SUMMARY
seed=  0 | ARI 0.085677 | NMI 0.124608 | runtime 61.00s
seed=  1 | ARI 0.087651 | NMI 0.110354 | runtime 57.84s
seed=  2 | ARI 0.063739 | NMI 0.101725 | runtime 59.26s
seed=100 | ARI 0.075202 | NMI 0.107190 | runtime 59.23s

ARI mean ± std = 0.078067 ± 0.009530
NMI mean ± std = 0.110969 ± 0.008458

Saved: /kaggle/working/MISO_oldenv_S2E18_diag

PASS: old-environment diagnostic complete.

Subprocess return code: 0


最终保存 Cell

In [22]:
# ============================================================
# MISO FINAL SAVE / ARCHIVE
# formal 50 runs + audit + old-env diagnostic
# ============================================================

from pathlib import Path
import shutil
import hashlib


WORK = Path("/kaggle/working")

FORMAL_ROOT = (
    WORK
    / "MISO_baseline"
    / "formal_10seeds"
)

OLDENV_DIAG = (
    WORK
    / "MISO_oldenv_S2E18_diag"
)

ARCHIVE_ROOT = (
    WORK
    / "MISO_5datasets_10seeds_FORMAL_FINAL"
)

ZIP_PATH = Path(
    str(ARCHIVE_ROOT) + ".zip"
)


assert FORMAL_ROOT.exists()
assert OLDENV_DIAG.exists()


# ------------------------------------------------------------
# Clean staging folder
# ------------------------------------------------------------

if ARCHIVE_ROOT.exists():
    shutil.rmtree(ARCHIVE_ROOT)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

ARCHIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Copy complete formal results
# ------------------------------------------------------------

shutil.copytree(
    FORMAL_ROOT,
    ARCHIVE_ROOT / "formal_10seeds",
)


# ------------------------------------------------------------
# Copy old-environment diagnostic
# ------------------------------------------------------------

shutil.copytree(
    OLDENV_DIAG,
    ARCHIVE_ROOT / "oldenv_S2E18_diagnostic",
)


# ------------------------------------------------------------
# README
# ------------------------------------------------------------

README = """MISO final archive

Formal experiment
=================
Datasets:
- HLN-A1
- HLN-D1
- E18.5
- S2-E15
- S2-E18

10 training seeds per dataset:
seed = 0..9

Total formal runs:
50

Formal protocol
===============
Official MISO implementation.

Views:
ind_views = all
combs = all
sparse = False

Preprocessing:
RNA:
filter_genes(min_cells=10) + log1p

ATAC:
filter_genes(min_cells=10) + log1p

Protein:
official MISO protein_norm

Training:
1000 epochs per modality
lr = 0.001

Embedding:
96 dimensions for two-modality data

Clustering:
KMeans
random_state = 100
n_init = 10

Evaluation:
ARI = adjusted_rand_score
NMI = normalized_mutual_info_score
      average_method = max
std = ddof=0

Formal status
=============
50/50 runs complete.
All datasets have seeds 0-9.
All runs reached exact target K.
Independent RAW/SUMMARY audit passed.

S2-E18 diagnostic
=================
Formal result:
ARI = 0.072740 +/- 0.010165
NMI = 0.108509 +/- 0.008849

Because this differed from a published reproduction,
S2-E18 was additionally tested under the historical
official software environment:

Python 3.7.13
NumPy 1.21.6
SciPy 1.7.3
scikit-learn 1.0.2
Scanpy 1.9.1
AnnData 0.8.0
PyTorch 1.13.1+cu117

Diagnostic seeds:
0, 1, 2, 100

Old-environment diagnostic:
ARI = 0.078067 +/- 0.009530
NMI = 0.110969 +/- 0.008458

The old environment therefore produced results consistent
with the formal modern-environment reproduction.

The old-environment diagnostic is diagnostic only and is
NOT included in the formal 10-seed benchmark.
"""


(
    ARCHIVE_ROOT
    / "README.txt"
).write_text(
    README,
    encoding="utf-8",
)


# ------------------------------------------------------------
# SHA256 manifest
# ------------------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with path.open("rb") as f:

        while True:
            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


manifest = []


for path in sorted(
    ARCHIVE_ROOT.rglob("*")
):

    if not path.is_file():
        continue

    if path.name == "SHA256SUMS.txt":
        continue

    rel = path.relative_to(
        ARCHIVE_ROOT
    )

    manifest.append(
        f"{sha256_file(path)}  {rel}"
    )


(
    ARCHIVE_ROOT
    / "SHA256SUMS.txt"
).write_text(
    "\n".join(manifest) + "\n",
    encoding="utf-8",
)


# ------------------------------------------------------------
# Basic structural checks
# ------------------------------------------------------------

formal_dir = (
    ARCHIVE_ROOT
    / "formal_10seeds"
)

assert len(
    list(
        formal_dir.glob(
            "*_seed*/metrics.json"
        )
    )
) == 50


assert (
    formal_dir
    / "MISO_5datasets_10seeds_RAW.csv"
).exists()

assert (
    formal_dir
    / "MISO_5datasets_10seeds_SUMMARY.csv"
).exists()

assert (
    formal_dir
    / "MISO_5datasets_10seeds_AUDIT_RAW.csv"
).exists()

assert (
    formal_dir
    / "MISO_5datasets_10seeds_AUDIT_SUMMARY.csv"
).exists()

assert (
    ARCHIVE_ROOT
    / "oldenv_S2E18_diagnostic"
    / "summary.json"
).exists()


# ------------------------------------------------------------
# ZIP
# ------------------------------------------------------------

zip_file = shutil.make_archive(
    str(ARCHIVE_ROOT),
    "zip",
    root_dir=ARCHIVE_ROOT.parent,
    base_dir=ARCHIVE_ROOT.name,
)


zip_file = Path(
    zip_file
)


print("=" * 100)
print("MISO FINAL ARCHIVE")
print("=" * 100)

print(
    "Formal runs:",
    len(
        list(
            formal_dir.glob(
                "*_seed*/metrics.json"
            )
        )
    )
)

print(
    "SHA256 entries:",
    len(manifest)
)

print(
    "\nZIP:"
)

print(
    zip_file
)

print(
    "\nZIP size:",
    f"{zip_file.stat().st_size / 1024 / 1024:.2f} MB"
)

print(
    "\nPASS: MISO final archive ready."
)

MISO FINAL ARCHIVE
Formal runs: 50
SHA256 entries: 323

ZIP:
/kaggle/working/MISO_5datasets_10seeds_FORMAL_FINAL.zip

ZIP size: 49.36 MB

PASS: MISO final archive ready.
